In [1]:
from pathlib import Path

ARTIFACTS = Path("../artifacts")
for f in sorted(ARTIFACTS.iterdir()):
    print(f.name)

.gitkeep
clean_data.csv
split_config.json


In [2]:
%pip install xgboost lightgbm catboost

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
import pandas as pd

df = pd.read_csv(ARTIFACTS / "clean_data.csv")

with open(ARTIFACTS / "split_config.json") as f:
    split_config = json.load(f)

print(df.shape)
print(df.columns.tolist())
print(split_config)

(526, 13)
['row_id', 'Surface Area (m2/g)', 'Total Pore Volume(cm3/g)', 'Micropore Volume (cm3/g)', 'C (%)', 'H (%)', 'N (%)', 'O (%)', 'S (%)', 'Temp (°C)', 'Pressure (bar)', 'CO2 uptake (mmol/g)', 'split']
{'random_state': 42, 'duplicates_removed': 1, 'counts': {'train': 336, 'test': 106, 'validation': 84}}


In [5]:
TARGET = "CO2 uptake (mmol/g)"
FEATURES = [c for c in df.columns if c not in ["row_id", TARGET, "split"]]

train = df[df["split"] == "train"]
X_train = train[FEATURES]
y_train = train[TARGET]

print(len(FEATURES), X_train.shape, y_train.shape)

10 (336, 10) (336,)


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

SEED = split_config["random_state"]

models = {
    "Random Forest":     RandomForestRegressor(n_estimators=500, random_state=SEED, n_jobs=-1),
    "Extra Trees":       ExtraTreesRegressor(n_estimators=500, random_state=SEED, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
    "XGBoost":           XGBRegressor(random_state=SEED, n_jobs=-1),
    "LightGBM":          LGBMRegressor(random_state=SEED, verbose=-1),
    "CatBoost":          CatBoostRegressor(random_state=SEED, verbose=0),
}

pipelines = {name: make_pipeline(SimpleImputer(strategy="median"), m)
             for name, m in models.items()}

In [7]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring=["neg_root_mean_squared_error", "r2"],
                            return_train_score=True)
    rows.append({
        "model":       name,
        "cv_rmse":     -scores["test_neg_root_mean_squared_error"].mean(),
        "cv_rmse_std":  scores["test_neg_root_mean_squared_error"].std(),
        "cv_r2":        scores["test_r2"].mean(),
        "train_rmse":  -scores["train_neg_root_mean_squared_error"].mean(),
    })

results = pd.DataFrame(rows).sort_values("cv_rmse").round(3)
results

,model,cv_rmse,cv_rmse_std,cv_r2,train_rmse
5,CatBoost,0.703,0.088,0.781,0.129
3,XGBoost,0.758,0.106,0.746,0.006
1,Extra Trees,0.765,0.081,0.741,0.000
0,Random Forest,0.779,0.064,0.733,0.281
4,LightGBM,0.779,0.080,0.733,0.334
2,Gradient Boosting,0.787,0.082,0.727,0.360


In [8]:
RESULTS = Path("../results")
results.to_csv(RESULTS / "04_model_comparison.csv", index=False)
print("saved")

saved
